# AIML-02 GPU Training Notebook (Colab / Kaggle)
> **FOR EDUCATION ONLY — NOT FOR CLINICAL USE.**

Run this notebook on a free Google Colab T4 GPU or Kaggle GPU to train both:
1. **2D Chest X-Ray Classifier (`DenseNet-121`)** on CheXpert / NIH ChestX-ray14 + Grad-CAM verification.
2. **3D Volumetric Segmentation (`MONAI 3D U-Net`)** on Medical Segmentation Decathlon (`Task06_Lung`).
3. Export `models/classifier_densenet121.pth`, `models/unet3d_monai.pth`, and `reports/metrics.json` back to your repo.

In [ ]:
!pip install -q torch torchvision monai pydicom nibabel scikit-learn pyyaml dvc
import torch, monai
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| MONAI:', monai.__version__)

In [ ]:
# Download Medical Segmentation Decathlon Task06_Lung from official AWS Open Data mirror
import os
os.makedirs('data/raw', exist_ok=True)
!aws s3 cp --no-sign-request s3://msd-for-monai/Task06_Lung.tar data/raw/Task06_Lung.tar || echo 'Install awscli or download from http://medicaldecathlon.com/'
!tar -xf data/raw/Task06_Lung.tar -C data/raw/ 2>/dev/null || true

In [ ]:
# Train 2D Classifier (DenseNet-121) and MONAI 3D U-Net using the repo scripts
!python -m ml.preprocess --params params.yaml --input data/raw --output data/processed
!python -m ml.train_classifier --params params.yaml --data data/processed --output models/classifier_densenet121.pth
!python -m ml.train_segmentation --params params.yaml --data data/processed --output models/unet3d_monai.pth
!python -m ml.evaluate --params params.yaml --data data/processed --cls-model models/classifier_densenet121.pth --seg-model models/unet3d_monai.pth --output reports/metrics.json

In [ ]:
# Inspect generated metrics and download checkpoints
import json
print(json.dumps(json.load(open('reports/metrics.json')), indent=2))